In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ================================
# 1. Load the Dataset
# ================================
file_path = "encoded_cricket_data.csv"  # Update with your actual file path
df = pd.read_csv(file_path)

# Selecting features and target variable
features = ["batting_team", "bowling_team", "inning", "cumulative_wickets","cumulative_runs",
            "current_run_rate", "required_run_rate", "venue", "target"]
target = "win"

# ================================
# 2. Train-Test Split
# ================================
X_train, X_test, y_train, y_test = train_test_split(df[features], df[target], test_size=0.3, random_state=42)

# Standardizing numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ================================
# 3. Hyperparameter Tuning for Random Forest
# ================================
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

# Best model
best_rf = grid_search.best_estimator_

# ================================
# 4. Model Evaluation
# ================================
y_train_pred = best_rf.predict(X_train_scaled)
y_test_pred = best_rf.predict(X_test_scaled)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("\n✅ Best Random Forest Parameters:", grid_search.best_params_)
print(f"✅ Random Forest Accuracy - Train: {train_accuracy:.4f}, Test: {test_accuracy:.4f}")
print("\n📊 Classification Report (Test Data):")
print(classification_report(y_test, y_test_pred))
print("\n🔍 Confusion Matrix (Test Data):")
print(confusion_matrix(y_test, y_test_pred))
print("\n" + "="*50)

# ================================
# 5. Feature Importance
# ================================
feature_importance = best_rf.feature_importances_
importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importance})
importance_df = importance_df.sort_values(by="Importance", ascending=False)

print("\n🚀 Feature Importance from Random Forest Model:")
print(importance_df)


/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



✅ Best Random Forest Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
✅ Random Forest Accuracy - Train: 0.9976, Test: 0.9804

📊 Classification Report (Test Data):
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     40052
           1       0.98      0.98      0.98     38224

    accuracy                           0.98     78276
   macro avg       0.98      0.98      0.98     78276
weighted avg       0.98      0.98      0.98     78276


🔍 Confusion Matrix (Test Data):
[[39323   729]
 [  803 37421]]


🚀 Feature Importance from Random Forest Model:
              Feature  Importance
7               venue    0.166760
5    current_run_rate    0.144322
1        bowling_team    0.131620
0        batting_team    0.128615
6   required_run_rate    0.115244
4     cumulative_runs    0.113944
8              target    0.111951
3  cumulative_wickets    0.075890
2              inning    0.011653


In [2]:
import joblib

# Save the trained model
joblib.dump(best_rf, "random_forest_model.pkl")
print("✅ Model saved as random_forest_model.pkl")


✅ Model saved as random_forest_model.pkl


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import joblib

# Define the path in your Google Drive
model_path = "/content/drive/My Drive/random_forest_model.pkl"

# Save the trained model
joblib.dump(best_rf, model_path)
print(f"✅ Model saved in Drive: {model_path}")


✅ Model saved in Drive: /content/drive/My Drive/random_forest_model.pkl


In [23]:
import pandas as pd
import numpy as np
import joblib
# ---------------------------
# User Input (example values)
# ---------------------------
# In a real scenario, these inputs would come from a user interface or a live data feed.
user_input = {
    'inning': 1,
    'cum_runs': 80,            # current cumulative runs
    'cum_wickets': 5,          # current wickets lost
    'overs_completed': 12.0,    # overs completed so far
    'target': 0,             # target score
    'batting_team': "Mumbai Indians",
    'bowling_team': "Chennai Super Kings",
    'venue': "Wankhede Stadium"  # assume this maps to a canonical venue (e.g., "Wankhede Stadium")
}

In [24]:
# ---------------------------
# 1. Compute Derived Features
# ---------------------------
# Compute current run rate
if user_input['overs_completed'] > 0:
    current_run_rate = user_input['cum_runs'] / user_input['overs_completed']
else:
    current_run_rate = 0

# Compute required run rate
remaining_overs = 20 - user_input['overs_completed']
if user_input['inning'] == 2 and remaining_overs > 0:
    required_run_rate = (user_input['target'] - user_input['cum_runs']) / remaining_overs
else:
    required_run_rate = 0

In [25]:
# ---------------------------
# 2. Load Saved Encoders and Model
# ---------------------------
# Load the team and venue LabelEncoders (ensure these files were saved during training)
le_team = joblib.load('le_team.pkl')
le_venue = joblib.load('le_venue.pkl')

# Load the final trained model (e.g., a Random Forest model)
model = joblib.load('/content/random_forest_model.pkl')


/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.1.3 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [26]:
# ---------------------------
# 3. Encode Categorical Variables
# ---------------------------
# Encode batting team and bowling team using the team encoder
batting_team_encoded = le_team.transform([user_input['batting_team']])[0]
bowling_team_encoded = le_team.transform([user_input['bowling_team']])[0]

# Encode the venue using the venue encoder
venue_canonical_encoded = le_venue.transform([user_input['venue']])[0]

# ---------------------------
# 4. Create Input DataFrame for Inference
# ---------------------------
# The final features (order matters) are:
# ['inning', 'cum_runs', 'cum_wickets', 'current_run_rate',
#  'required_run_rate', 'target', 'batting_team_encoded',
#  'bowling_team_encoded', 'venue_canonical_encoded']
input_data = {
    'inning': [user_input['inning']],
    'cum_runs': [user_input['cum_runs']],
    'cum_wickets': [user_input['cum_wickets']],
    'current_run_rate': [current_run_rate],
    'required_run_rate': [required_run_rate],
    'target': [user_input['target']],
    'batting_team_encoded': [batting_team_encoded],
    'bowling_team_encoded': [bowling_team_encoded],
    'venue_canonical_encoded': [venue_canonical_encoded]
}

input_df = pd.DataFrame(input_data)

In [27]:
# ---------------------------
# 5. Make Prediction
# ---------------------------
# The model predicts a binary outcome: 1 indicates the batting team wins, 0 indicates the bowling team wins.
prediction = model.predict(input_df)[0]
predicted_probabilities = model.predict_proba(input_df)[0]

# ---------------------------
# 6. Map Prediction to Team Names
# ---------------------------
if prediction == 1:
    predicted_winner = user_input['batting_team']
else:
    predicted_winner = user_input['bowling_team']

print("Predicted Winner:", predicted_winner)
print("Prediction Probabilities (Loss, Win):", predicted_probabilities)

Predicted Winner: Chennai Super Kings
Prediction Probabilities (Loss, Win): [0.995 0.005]


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
